# Orthogonality and Orthonormality - Examples

This notebook demonstrates orthogonality concepts with practical implementations.

## Contents
1. Orthogonality Basics
2. Orthonormal Vectors
3. Orthogonal Matrices
4. Gram-Schmidt Process
5. Orthogonal Projections
6. Projection onto Subspaces
7. QR Decomposition
8. Least Squares via QR
9. Orthogonal Complement
10. Orthogonal Weight Initialization
11. PCA and Orthogonality
12. Householder Reflections
13. Visualizations

In [ ]:
import numpy as np
from numpy.linalg import norm, inv, qr, det, eig, svd
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

## 1. Orthogonality Basics

Two vectors $\mathbf{u}$ and $\mathbf{v}$ are **orthogonal** (perpendicular) if:

$$\mathbf{u} \cdot \mathbf{v} = \mathbf{u}^T \mathbf{v} = \sum_{i=1}^n u_i v_i = 0$$

Geometrically: they form a 90° angle.

In [ ]:
# Basic orthogonality check
u = np.array([1, 0, 0])
v = np.array([0, 1, 0])
w = np.array([1, 1, 0])

print(f"u = {u}")
print(f"v = {v}")
print(f"w = {w}")

print(f"\nu · v = {np.dot(u, v)} → Orthogonal: {np.dot(u, v) == 0}")
print(f"u · w = {np.dot(u, w)} → Orthogonal: {np.dot(u, w) == 0}")

In [ ]:
# Pythagorean Theorem for orthogonal vectors
# If u ⊥ v, then ||u + v||² = ||u||² + ||v||²

print("Pythagorean Theorem (u ⊥ v):")
print(f"||u||² + ||v||² = {norm(u)**2} + {norm(v)**2} = {norm(u)**2 + norm(v)**2}")
print(f"||u + v||² = ||{u + v}||² = {norm(u + v)**2}")
print(f"Equal: {np.isclose(norm(u)**2 + norm(v)**2, norm(u + v)**2)} ✓")

print("\nFor non-orthogonal u and w:")
print(f"||u||² + ||w||² = {norm(u)**2} + {norm(w)**2} = {norm(u)**2 + norm(w)**2}")
print(f"||u + w||² = ||{u + w}||² = {norm(u + w)**2}")
print(f"Not equal (cross-term 2u·w = {2*np.dot(u, w)})")

## 2. Orthonormal Vectors

Vectors are **orthonormal** if they are:
1. **Orthogonal**: $\mathbf{u}_i \cdot \mathbf{u}_j = 0$ for $i \neq j$
2. **Normalized**: $\|\mathbf{u}_i\| = 1$ for all $i$

Compact form using Kronecker delta:
$$\mathbf{u}_i^T \mathbf{u}_j = \delta_{ij} = \begin{cases} 1 & i = j \\ 0 & i \neq j \end{cases}$$

In [ ]:
# Standard basis (orthonormal)
e1 = np.array([1, 0, 0])
e2 = np.array([0, 1, 0])
e3 = np.array([0, 0, 1])

print("Standard basis vectors:")
print(f"e1 = {e1}, e2 = {e2}, e3 = {e3}")

print("\nOrthogonality:")
print(f"e1 · e2 = {np.dot(e1, e2)}")
print(f"e1 · e3 = {np.dot(e1, e3)}")
print(f"e2 · e3 = {np.dot(e2, e3)}")

print("\nNormalization:")
print(f"||e1|| = {norm(e1)}, ||e2|| = {norm(e2)}, ||e3|| = {norm(e3)}")

In [ ]:
# Custom orthonormal basis (45° rotation in xy-plane)
s = 1 / np.sqrt(2)
u1 = np.array([s, s, 0])
u2 = np.array([-s, s, 0])
u3 = np.array([0, 0, 1])

print("Custom orthonormal basis (45° rotated):")
print(f"u1 = {np.round(u1, 4)}")
print(f"u2 = {np.round(u2, 4)}")
print(f"u3 = {u3}")

print("\nVerification:")
print(f"u1 · u2 = {np.dot(u1, u2):.10f} ≈ 0")
print(f"||u1|| = {norm(u1):.4f} = 1")
print(f"||u2|| = {norm(u2):.4f} = 1")

## 3. Orthogonal Matrices

A square matrix $Q$ is **orthogonal** if:

$$Q^T Q = Q Q^T = I$$

Equivalently: $Q^{-1} = Q^T$ (inverse is just transpose!)

**Key insight**: The columns of $Q$ form an orthonormal basis.

In [ ]:
# Rotation matrix (orthogonal)
theta = np.pi / 4  # 45 degrees
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta), np.cos(theta)]])

print(f"Rotation matrix R (45°):")
print(R)

print(f"\nR^T @ R = ")
print(np.round(R.T @ R, 10))
print(f"Is orthogonal: {np.allclose(R.T @ R, np.eye(2))}")

In [ ]:
# Properties of orthogonal matrices
x = np.array([3, 4])
Rx = R @ x

# Property 1: Inverse = Transpose
print("1. R⁻¹ = R^T:")
print(f"   R⁻¹ @ R = \n{np.round(inv(R) @ R, 4)}")
print(f"   R^T @ R = \n{np.round(R.T @ R, 4)}")

# Property 2: Preserves length
print(f"\n2. Length preservation:")
print(f"   ||x|| = {norm(x)}")
print(f"   ||Rx|| = {norm(Rx):.4f}")

# Property 3: Determinant = ±1
print(f"\n3. Determinant:")
print(f"   det(R) = {det(R):.4f} (rotation: +1)")

# Reflection matrix
M = np.array([[1, 0], [0, -1]])  # Reflect across x-axis
print(f"   det(M) = {det(M):.0f} (reflection: -1)")

In [ ]:
# Property 4: Preserves angles (dot products)
y = np.array([1, 0])
Ry = R @ y

cos_before = np.dot(x, y) / (norm(x) * norm(y))
cos_after = np.dot(Rx, Ry) / (norm(Rx) * norm(Ry))

print("4. Angle preservation:")
print(f"   cos(angle before) = {cos_before:.4f}")
print(f"   cos(angle after) = {cos_after:.4f}")

# Property 5: Product of orthogonal = orthogonal
R2 = np.array([[np.cos(np.pi/6), -np.sin(np.pi/6)],
               [np.sin(np.pi/6), np.cos(np.pi/6)]])
RR2 = R @ R2
print(f"\n5. Product of orthogonal matrices:")
print(f"   (R @ R2)^T @ (R @ R2) = \n{np.round(RR2.T @ RR2, 4)}")
print(f"   Still orthogonal: {np.allclose(RR2.T @ RR2, np.eye(2))}")

## 4. Gram-Schmidt Process

Convert any linearly independent set into an orthonormal set:

**Algorithm**:
1. $\mathbf{u}_1 = \frac{\mathbf{v}_1}{\|\mathbf{v}_1\|}$
2. $\mathbf{w}_k = \mathbf{v}_k - \sum_{j=1}^{k-1} (\mathbf{v}_k \cdot \mathbf{u}_j)\mathbf{u}_j$, then $\mathbf{u}_k = \frac{\mathbf{w}_k}{\|\mathbf{w}_k\|}$

**Intuition**: Each step subtracts the projections onto previous vectors.

In [ ]:
# Original vectors (linearly independent but not orthogonal)
v1 = np.array([1, 1, 0], dtype=float)
v2 = np.array([1, 0, 1], dtype=float)
v3 = np.array([0, 1, 1], dtype=float)

print("Original vectors:")
print(f"v1 = {v1}")
print(f"v2 = {v2}")
print(f"v3 = {v3}")
print(f"\nv1 · v2 = {np.dot(v1, v2)} (not orthogonal)")

In [ ]:
# Step 1: Normalize v1
u1 = v1 / norm(v1)
print("Step 1: u1 = v1 / ||v1||")
print(f"u1 = {np.round(u1, 4)}")

# Step 2: Make v2 orthogonal to u1, then normalize
proj_u1_v2 = np.dot(v2, u1) * u1
w2 = v2 - proj_u1_v2
u2 = w2 / norm(w2)

print("\nStep 2:")
print(f"  proj_u1(v2) = (v2 · u1) × u1 = {np.dot(v2, u1):.4f} × u1 = {np.round(proj_u1_v2, 4)}")
print(f"  w2 = v2 - proj = {np.round(w2, 4)}")
print(f"  u2 = w2 / ||w2|| = {np.round(u2, 4)}")

# Step 3: Make v3 orthogonal to u1 and u2, then normalize
proj_u1_v3 = np.dot(v3, u1) * u1
proj_u2_v3 = np.dot(v3, u2) * u2
w3 = v3 - proj_u1_v3 - proj_u2_v3
u3 = w3 / norm(w3)

print("\nStep 3:")
print(f"  w3 = v3 - proj_u1(v3) - proj_u2(v3)")
print(f"  w3 = {np.round(w3, 4)}")
print(f"  u3 = {np.round(u3, 4)}")

In [ ]:
# Verify orthonormality
print("Verification:")
print(f"u1 · u2 = {np.dot(u1, u2):.10f} ≈ 0 ✓")
print(f"u1 · u3 = {np.dot(u1, u3):.10f} ≈ 0 ✓")
print(f"u2 · u3 = {np.dot(u2, u3):.10f} ≈ 0 ✓")
print(f"\n||u1|| = {norm(u1):.4f}, ||u2|| = {norm(u2):.4f}, ||u3|| = {norm(u3):.4f}")

# Form Q matrix
Q = np.column_stack([u1, u2, u3])
print(f"\nQ = [u1 | u2 | u3]:\n{np.round(Q, 4)}")
print(f"\nQ^T @ Q = \n{np.round(Q.T @ Q, 4)}")

In [ ]:
# Gram-Schmidt as a function
def gram_schmidt(V):
    """Classical Gram-Schmidt orthogonalization."""
    n, k = V.shape
    Q = np.zeros((n, k))
    
    for j in range(k):
        v = V[:, j].copy()
        for i in range(j):
            v -= np.dot(Q[:, i], V[:, j]) * Q[:, i]
        Q[:, j] = v / norm(v)
    
    return Q

def modified_gram_schmidt(V):
    """Modified Gram-Schmidt (more numerically stable)."""
    n, k = V.shape
    Q = V.copy().astype(float)
    
    for j in range(k):
        Q[:, j] = Q[:, j] / norm(Q[:, j])
        for i in range(j + 1, k):
            Q[:, i] -= np.dot(Q[:, j], Q[:, i]) * Q[:, j]
    
    return Q

# Test
V = np.column_stack([v1, v2, v3])
Q_classical = gram_schmidt(V)
Q_modified = modified_gram_schmidt(V)

print("Classical vs Modified Gram-Schmidt:")
print(f"Both produce orthonormal: {np.allclose(Q_classical.T @ Q_classical, np.eye(3))}")
print("(Modified is preferred for numerical stability)")

## 5. Orthogonal Projections

The projection of $\mathbf{v}$ onto $\mathbf{u}$:

$$\text{proj}_{\mathbf{u}}(\mathbf{v}) = \frac{\mathbf{v} \cdot \mathbf{u}}{\|\mathbf{u}\|^2} \mathbf{u}$$

If $\mathbf{u}$ is a unit vector: $\text{proj}_{\mathbf{u}}(\mathbf{v}) = (\mathbf{v} \cdot \mathbf{u}) \mathbf{u}$

In [ ]:
# Projection onto a vector
u = np.array([1, 2])
v = np.array([3, 1])

print(f"u = {u}, v = {v}")

# Calculate projection
proj = (np.dot(v, u) / np.dot(u, u)) * u
print(f"\nproj_u(v) = (v·u / ||u||²) × u")
print(f"         = ({np.dot(v, u)} / {np.dot(u, u)}) × {u}")
print(f"         = {proj}")

# Orthogonal component
perp = v - proj
print(f"\nOrthogonal component: v_⊥ = v - proj = {perp}")

# Verify orthogonality
print(f"\nVerify: proj · v_⊥ = {np.dot(proj, perp):.10f} ≈ 0 ✓")

In [ ]:
# Projection matrix P = uu^T / (u^T u)
P = np.outer(u, u) / np.dot(u, u)
print(f"Projection matrix P = uu^T / (u^T u):")
print(P)

print(f"\nP @ v = {P @ v} (same as proj!)")

# Properties of projection matrices
print("\nProjection matrix properties:")
print(f"1. P² = P (idempotent): {np.allclose(P @ P, P)}")
print(f"2. P^T = P (symmetric): {np.allclose(P.T, P)}")
print(f"3. rank(P) = 1: {np.linalg.matrix_rank(P)}")

# Complementary projection
P_perp = np.eye(2) - P
print(f"\nComplementary projection I - P:")
print(P_perp)
print(f"(I - P) @ v = {P_perp @ v} (orthogonal component!)")

## 6. Projection onto Subspaces

If columns of $A$ span a subspace, the projection matrix is:

$$P = A(A^T A)^{-1} A^T$$

If columns of $A$ are already orthonormal ($Q$):
$$P = QQ^T$$

In [ ]:
# Subspace spanned by columns of A
A = np.array([[1, 0],
              [1, 1],
              [0, 1]], dtype=float)

print(f"Subspace basis (columns of A):\n{A}")

# Vector to project
b = np.array([1, 2, 3], dtype=float)
print(f"\nVector b = {b}")

# Projection matrix P = A(A^T A)^-1 A^T
P = A @ inv(A.T @ A) @ A.T
print(f"\nProjection matrix P = A(A^TA)⁻¹A^T:")
print(np.round(P, 4))

# Project b
proj_b = P @ b
print(f"\nProjection of b: P @ b = {np.round(proj_b, 4)}")

# Error vector
error = b - proj_b
print(f"Error vector: b - Pb = {np.round(error, 4)}")

# Verify: error is orthogonal to column space
print(f"\nError ⊥ column space: A^T @ error = {np.round(A.T @ error, 10)}")

In [ ]:
# Using orthonormal basis (much simpler!)
Q_sub, R_sub = qr(A)
Q_sub = Q_sub[:, :2]  # Thin Q

print(f"Q (orthonormal columns):\n{np.round(Q_sub, 4)}")

P_qr = Q_sub @ Q_sub.T
print(f"\nP = QQ^T:\n{np.round(P_qr, 4)}")
print(f"\nSame as before: {np.allclose(P, P_qr)}")

## 7. QR Decomposition

Any matrix $A$ with linearly independent columns can be factored:

$$A = QR$$

where:
- $Q$ has orthonormal columns
- $R$ is upper triangular with positive diagonal

**QR IS Gram-Schmidt in matrix form!**

In [ ]:
A = np.array([[1, 1],
              [1, 0],
              [0, 1]], dtype=float)

print(f"Matrix A:\n{A}")

# Manual QR via Gram-Schmidt
print("\n--- Manual QR via Gram-Schmidt ---")
a1, a2 = A[:, 0], A[:, 1]

# Orthonormalize
q1 = a1 / norm(a1)
w2 = a2 - np.dot(a2, q1) * q1
q2 = w2 / norm(w2)

Q = np.column_stack([q1, q2])
print(f"Q (orthonormal columns):\n{np.round(Q, 4)}")

# R = Q^T A
R = Q.T @ A
print(f"\nR (upper triangular) = Q^T A:\n{np.round(R, 4)}")

# Verify
print(f"\nVerify Q @ R = A:\n{np.round(Q @ R, 4)}")

In [ ]:
# NumPy QR
print("--- NumPy QR ---")
Q_np, R_np = qr(A)
print(f"Q:\n{np.round(Q_np[:, :2], 4)}")
print(f"\nR:\n{np.round(R_np[:2, :], 4)}")

# Note: NumPy may flip signs (both valid)
print(f"\nQ^T @ Q = \n{np.round(Q_np[:, :2].T @ Q_np[:, :2], 4)}")

## 8. Least Squares via QR

For overdetermined system $A\mathbf{x} = \mathbf{b}$:

**Normal equations**: $A^T A \hat{\mathbf{x}} = A^T \mathbf{b}$

**QR method** (more stable): If $A = QR$, then $\hat{\mathbf{x}} = R^{-1}Q^T\mathbf{b}$

In [ ]:
# Overdetermined system (linear regression)
A = np.array([[1, 1],
              [1, 2],
              [1, 3],
              [1, 4]], dtype=float)
b = np.array([2, 3, 4, 5.5], dtype=float)

print("Fitting y = β₀ + β₁x to points:")
print("x: 1, 2, 3, 4")
print("y: 2, 3, 4, 5.5")

# Via normal equations
print("\n--- Normal equations: (A^TA)x = A^Tb ---")
x_normal = inv(A.T @ A) @ A.T @ b
print(f"x = {np.round(x_normal, 4)}")

# Via QR
print("\n--- QR method: Rx = Q^Tb ---")
Q, R = qr(A)
Q = Q[:, :2]
R = R[:2, :]

x_qr = np.linalg.solve(R, Q.T @ b)
print(f"x = {np.round(x_qr, 4)}")

print(f"\nBoth methods agree: {np.allclose(x_normal, x_qr)}")
print(f"\nFitted line: y = {x_qr[0]:.4f} + {x_qr[1]:.4f}x")

In [ ]:
# Residual analysis
residual = b - A @ x_qr
print(f"Residual: b - Ax = {np.round(residual, 4)}")
print(f"||residual|| = {norm(residual):.4f}")

# Key property: residual is orthogonal to column space
print(f"\nResidual ⊥ columns of A:")
print(f"A^T @ residual = {np.round(A.T @ residual, 10)} ≈ 0 ✓")

## 9. Orthogonal Complement

For subspace $W$:

$$W^{\perp} = \{\mathbf{v} : \mathbf{v} \cdot \mathbf{w} = 0 \text{ for all } \mathbf{w} \in W\}$$

Properties:
- $\dim(W) + \dim(W^{\perp}) = n$
- $(W^{\perp})^{\perp} = W$
- $W \cap W^{\perp} = \{\mathbf{0}\}$

In [ ]:
# W = xy-plane (z = 0)
print("W = xy-plane (z = 0)")
w1 = np.array([1, 0, 0])
w2 = np.array([0, 1, 0])
print(f"Basis for W: w1 = {w1}, w2 = {w2}")

# Orthogonal complement W^perp = z-axis
print(f"\nW^⊥ = z-axis")
w_perp = np.array([0, 0, 1])
print(f"Basis for W^⊥: {w_perp}")

# Verify orthogonality
print(f"\nw1 · w_⊥ = {np.dot(w1, w_perp)}")
print(f"w2 · w_⊥ = {np.dot(w2, w_perp)}")

# Decompose a vector
v = np.array([3, 4, 5])
print(f"\nDecompose v = {v}:")
v_W = np.array([v[0], v[1], 0])  # In W (xy-plane)
v_perp = np.array([0, 0, v[2]])  # In W^perp (z-axis)

print(f"v_W (in xy-plane) = {v_W}")
print(f"v_⊥ (along z-axis) = {v_perp}")
print(f"v = v_W + v_⊥: {np.allclose(v, v_W + v_perp)}")
print(f"v_W ⊥ v_⊥: {np.dot(v_W, v_perp) == 0}")

# Dimension check
print(f"\ndim(W) + dim(W^⊥) = 2 + 1 = 3 = dim(ℝ³) ✓")

## 10. Orthogonal Weight Initialization

In neural networks, orthogonal initialization helps:
- Preserve signal magnitude through layers
- Prevent vanishing/exploding gradients
- Better training dynamics

In [ ]:
np.random.seed(42)

# Random initialization
W_random = np.random.randn(4, 4) * 0.1

# Orthogonal initialization (via QR)
M = np.random.randn(4, 4)
W_orth, _ = qr(M)

print("Random initialization:")
print(f"W @ W^T:\n{np.round(W_random @ W_random.T, 4)}")
print(f"Singular values: {np.round(svd(W_random)[1], 4)}")

print("\nOrthogonal initialization:")
print(f"W @ W^T:\n{np.round(W_orth @ W_orth.T, 4)}")
print(f"Singular values: {np.round(svd(W_orth)[1], 4)}")

In [ ]:
# Signal propagation through many layers
x = np.random.randn(4)
n_layers = 50

signal_random = x.copy()
signal_orth = x.copy()

norms_random = [norm(signal_random)]
norms_orth = [norm(signal_orth)]

for i in range(n_layers):
    signal_random = W_random @ signal_random
    signal_orth = W_orth @ signal_orth
    norms_random.append(norm(signal_random))
    norms_orth.append(norm(signal_orth))

print(f"After {n_layers} layers:")
print(f"Random: ||x|| = {norm(x):.2f} → ||W^{n_layers}x|| = {norms_random[-1]:.2e}")
print(f"Orthogonal: ||x|| = {norm(x):.2f} → ||W^{n_layers}x|| = {norms_orth[-1]:.2f}")

plt.figure(figsize=(10, 4))
plt.semilogy(norms_random, 'r-', label='Random init', linewidth=2)
plt.semilogy(norms_orth, 'b-', label='Orthogonal init', linewidth=2)
plt.axhline(y=norm(x), color='gray', linestyle='--', label='Original norm')
plt.xlabel('Layer')
plt.ylabel('Signal Norm (log scale)')
plt.title('Signal Propagation: Random vs Orthogonal Initialization')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 11. PCA and Orthogonality

PCA finds orthogonal directions of maximum variance.
The principal components (eigenvectors of covariance) are orthogonal!

In [ ]:
np.random.seed(42)

# Generate correlated data
n = 200
X = np.random.randn(n, 2)
X[:, 1] = 0.8 * X[:, 0] + 0.2 * X[:, 1]  # Add correlation
X = X - X.mean(axis=0)  # Center

# Covariance matrix
C = X.T @ X / (n - 1)
print("Covariance matrix:")
print(np.round(C, 4))

# Eigendecomposition
eigenvalues, eigenvectors = eig(C)
idx = eigenvalues.argsort()[::-1]
eigenvalues = eigenvalues[idx].real
eigenvectors = eigenvectors[:, idx].real

print(f"\nEigenvalues: {np.round(eigenvalues, 4)}")
print(f"\nPrincipal components:")
print(f"PC1 = {np.round(eigenvectors[:, 0], 4)}")
print(f"PC2 = {np.round(eigenvectors[:, 1], 4)}")
print(f"\nPC1 · PC2 = {np.round(np.dot(eigenvectors[:, 0], eigenvectors[:, 1]), 10)}")
print("Principal components are orthogonal!")

In [ ]:
# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Original data
ax = axes[0]
ax.scatter(X[:, 0], X[:, 1], alpha=0.5, s=20)
# Plot principal components
scale = 2
ax.arrow(0, 0, scale*eigenvectors[0, 0], scale*eigenvectors[1, 0], 
         color='red', width=0.05, head_width=0.15, label='PC1')
ax.arrow(0, 0, scale*eigenvectors[0, 1], scale*eigenvectors[1, 1], 
         color='blue', width=0.05, head_width=0.15, label='PC2')
ax.set_xlabel('X1')
ax.set_ylabel('X2')
ax.set_title('Original Data + Principal Components')
ax.set_aspect('equal')
ax.legend()
ax.grid(True, alpha=0.3)

# Transformed data
X_pca = X @ eigenvectors
ax = axes[1]
ax.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.5, s=20)
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_title('Data in PCA Coordinates (Decorrelated)')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Covariance in PCA basis (should be diagonal!)
C_pca = np.cov(X_pca.T)
print("Covariance in PCA basis (diagonal!):")
print(np.round(C_pca, 4))

## 12. Householder Reflections

More numerically stable than Gram-Schmidt for QR.

A Householder reflection:
$$H = I - 2\mathbf{v}\mathbf{v}^T$$

where $\|\mathbf{v}\| = 1$. This reflects vectors across the hyperplane perpendicular to $\mathbf{v}$.

In [ ]:
def householder(a):
    """Compute Householder vector and matrix to zero out below first element."""
    n = len(a)
    e1 = np.zeros(n)
    e1[0] = 1
    
    # v = a + sign(a[0]) * ||a|| * e1
    v = a + np.sign(a[0]) * norm(a) * e1
    v = v / norm(v)
    
    # H = I - 2vv^T
    H = np.eye(n) - 2 * np.outer(v, v)
    
    return H, v

# Example
a = np.array([3, 1, 2], dtype=float)
print(f"Original vector: a = {a}")

H, v = householder(a)
print(f"\nHouseholder matrix H:")
print(np.round(H, 4))

Ha = H @ a
print(f"\nH @ a = {np.round(Ha, 4)}")
print("(All elements below first are zeroed!)")

# H is orthogonal
print(f"\nH is orthogonal: {np.allclose(H.T @ H, np.eye(3))}")
print(f"H is symmetric: {np.allclose(H, H.T)}")
print(f"H² = I: {np.allclose(H @ H, np.eye(3))} (reflection)")

## 13. Visualizations

In [ ]:
# Gram-Schmidt visualization
v1 = np.array([2, 1])
v2 = np.array([1, 2])

# Gram-Schmidt
u1 = v1 / norm(v1)
proj = np.dot(v2, u1) * u1
w2 = v2 - proj
u2 = w2 / norm(w2)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Original
ax = axes[0]
ax.quiver(0, 0, v1[0], v1[1], angles='xy', scale_units='xy', scale=1, 
          color='blue', width=0.03, label='v₁')
ax.quiver(0, 0, v2[0], v2[1], angles='xy', scale_units='xy', scale=1, 
          color='red', width=0.03, label='v₂')
ax.set_xlim(-0.5, 3)
ax.set_ylim(-0.5, 2.5)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=10)
ax.set_title('Original Vectors\n(not orthogonal)', fontsize=11)

# Step 1: Subtract projection
ax = axes[1]
ax.quiver(0, 0, u1[0], u1[1], angles='xy', scale_units='xy', scale=1, 
          color='blue', width=0.03, label='u₁ (normalized v₁)')
ax.quiver(0, 0, v2[0], v2[1], angles='xy', scale_units='xy', scale=1, 
          color='red', alpha=0.4, width=0.02, label='v₂')
ax.quiver(0, 0, proj[0], proj[1], angles='xy', scale_units='xy', scale=1, 
          color='green', width=0.03, label='proj')
ax.quiver(proj[0], proj[1], w2[0], w2[1], angles='xy', scale_units='xy', scale=1, 
          color='orange', width=0.03, label='w₂ = v₂ - proj')
ax.set_xlim(-0.5, 3)
ax.set_ylim(-0.5, 2.5)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)
ax.set_title('Subtract Projection\nv₂ - proj_{u₁}(v₂)', fontsize=11)

# Result
ax = axes[2]
ax.quiver(0, 0, u1[0], u1[1], angles='xy', scale_units='xy', scale=1, 
          color='blue', width=0.04, label='u₁')
ax.quiver(0, 0, u2[0], u2[1], angles='xy', scale_units='xy', scale=1, 
          color='red', width=0.04, label='u₂')
# Draw right angle
ax.plot([0.1, 0.1, 0.0], [0.0, 0.1, 0.1], 'k-', linewidth=1)
ax.set_xlim(-0.5, 3)
ax.set_ylim(-0.5, 2.5)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=10)
ax.set_title('Result: Orthonormal\nu₁ ⊥ u₂, ||u|| = 1', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# Projection visualization
fig, ax = plt.subplots(figsize=(8, 6))

u = np.array([3, 1])
v = np.array([1, 2.5])
proj = (np.dot(v, u) / np.dot(u, u)) * u
perp = v - proj

# Draw vectors
ax.quiver(0, 0, u[0], u[1], angles='xy', scale_units='xy', scale=1, 
          color='blue', width=0.03, label='u')
ax.quiver(0, 0, v[0], v[1], angles='xy', scale_units='xy', scale=1, 
          color='red', width=0.03, label='v')
ax.quiver(0, 0, proj[0], proj[1], angles='xy', scale_units='xy', scale=1, 
          color='green', width=0.03, label='proj_u(v)')
ax.quiver(proj[0], proj[1], perp[0], perp[1], angles='xy', scale_units='xy', scale=1, 
          color='purple', width=0.03, label='v - proj (⊥ to u)')

# Dashed line from v to proj
ax.plot([v[0], proj[0]], [v[1], proj[1]], 'k--', alpha=0.5)

# Right angle marker
angle_size = 0.15
ax.plot([proj[0], proj[0] - angle_size * perp[1]/norm(perp)],
        [proj[1], proj[1] + angle_size * perp[0]/norm(perp)], 'k-', linewidth=1)
ax.plot([proj[0] - angle_size * perp[1]/norm(perp), 
         proj[0] - angle_size * perp[1]/norm(perp) + angle_size * u[0]/norm(u)],
        [proj[1] + angle_size * perp[0]/norm(perp), 
         proj[1] + angle_size * perp[0]/norm(perp) + angle_size * u[1]/norm(u)], 'k-', linewidth=1)

ax.set_xlim(-0.5, 4)
ax.set_ylim(-0.5, 3)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.legend(loc='upper left', fontsize=10)
ax.set_title('Orthogonal Projection', fontsize=12)
ax.set_xlabel('x')
ax.set_ylabel('y')

plt.tight_layout()
plt.show()

## Summary

| Concept | Definition | Key Property |
|---------|------------|--------------|
| Orthogonal vectors | $\mathbf{u} \cdot \mathbf{v} = 0$ | Perpendicular |
| Orthonormal | Orthogonal + unit length | $\mathbf{u}_i^T \mathbf{u}_j = \delta_{ij}$ |
| Orthogonal matrix | $Q^TQ = I$ | $Q^{-1} = Q^T$ |
| Projection | $\text{proj}_\mathbf{u}(\mathbf{v}) = \frac{\mathbf{v} \cdot \mathbf{u}}{\|\mathbf{u}\|^2}\mathbf{u}$ | Closest point on line |
| Gram-Schmidt | Orthogonalizes vectors | Creates orthonormal basis |
| QR decomposition | $A = QR$ | Q orthonormal, R upper triangular |

### Why Orthogonality Matters in ML

```
Orthogonal computations are:
├── Numerically stable (errors don't accumulate)
├── Computationally efficient (Q⁻¹ = Qᵀ)
├── Geometrically interpretable (rotations, reflections)
└── Mathematically elegant (preservation properties)

Applications:
├── PCA: Principal components are orthogonal
├── Weight init: Orthogonal matrices preserve gradients
├── SVD: U and V are orthogonal
├── Least squares: Error orthogonal to column space
└── Whitening: Decorrelate = make features orthogonal
```